# Diagnóstico de calidad de datos: criminalidad en Medellín

**Evidencia:** GA1-220501094-01-AA1-EV06
**Aprendiz:** Liliana Sofia Panesso Mendez

## Momento 1. Reconocimiento del proyecto y de los datasets

**Proyecto:** Análisis de los delitos más frecuentes en Medellín y su variación a través del tiempo.

**Propósito:** Conocer el comportamiento de la criminalidad en Medellín a partir de registros oficiales verificables, y no solo de percepciones, identificando qué delitos tienen más casos registrados y cómo ha sido su tendencia. Antes de iniciar ese análisis, este notebook diagnostica la calidad de los datasets seleccionados para determinar si están en condiciones adecuadas de uso y qué acciones de preparación requieren.

**Pregunta analítica:** ¿Cuáles son los delitos con mayor cantidad de casos registrados en Medellín y cómo ha variado su número de casos entre 2020 y 2025?

### Datasets seleccionados

| # | Dataset | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|---|
| 1 | criminalidad_anio_mes.csv | MEData, Alcaldía de Medellín (Secretaría de Seguridad y Convivencia). Republicado en Datos Abiertos Colombia. Licencia CC BY-SA 4.0 | Cantidad de casos de 7 conductas delictivas (extorsión, homicidio y hurtos a persona, de carro, de moto, a residencia y de semoviente) por año y mes, desde 2003. Columnas: `Fecha_hecho`, `Conducta`, `Cantidad_casos` | **Dataset principal.** Permite identificar los delitos con más casos y analizar su variación entre 2020 y 2025 |
| 2 | criminalidad_comunas_anio.csv | MEData, Alcaldía de Medellín (Secretaría de Seguridad y Convivencia). Republicado en Datos Abiertos Colombia. Licencia CC BY-SA 4.0 | Cantidad de casos de las mismas 7 conductas por comuna y año, desde 2003. Columnas: `Fecha_hecho`, `Conducta`, `Codigo_comuna`, `Cantidad_casos` | **Dataset complementario.** Aporta el lugar de ocurrencia para comparar los delitos entre zonas de la ciudad y contrastar los totales con el dataset 1 |
| 3 | homicidios_banco_mundial.csv | Banco Mundial, Indicadores del desarrollo mundial (datos de UNODC). Archivo original: `API_VC.IHR.PSRC.P5_DS2_es_csv_v2_13913.csv` | Tasa de homicidios intencionales por cada 100.000 habitantes, por país y región, con una columna por año desde 1960 | **Dataset de contexto.** No responde la pregunta por sí solo, pero permite ubicar la tendencia de homicidios de Colombia en el mismo período |

### Aspectos de calidad a revisar

1. **Cobertura temporal:** verificar que los años 2020 a 2025 estén completos en cada dataset, especialmente 2025.
2. **Valores faltantes:** identificar celdas vacías y valores faltantes "disfrazados" de texto, como `SIN DATO` en la columna de comunas.
3. **Registros duplicados:** comprobar que no haya filas repetidas que inflen los conteos.
4. **Tipos de datos:** revisar que las fechas, los años y las cantidades tengan un tipo coherente con lo que representan.
5. **Consistencia de categorías:** confirmar que los nombres de las conductas estén escritos de la misma forma dentro de cada dataset y entre los datasets 1 y 2.
6. **Valores incoherentes:** buscar cantidades negativas, años fuera de rango o códigos de comuna inválidos.
7. **Coherencia entre datasets:** comparar el formato de `Fecha_hecho` (año y mes frente a solo año) y verificar si los totales por comuna coinciden con los totales del dataset 1.
8. **Formato del archivo:** el archivo del Banco Mundial trae líneas de metadatos antes del encabezado, una columna por año y una columna final vacía.
9. **Comparabilidad de la medida:** el Banco Mundial reporta una tasa nacional por 100.000 habitantes, mientras que los datasets de Medellín reportan cantidad de casos.
10. **Limitación de origen:** los registros corresponden a delitos denunciados, por lo que existe subregistro de los casos no reportados.

## Momento 2. Carga de los datasets

In [1]:
import pandas as pd

crim_mes = pd.read_csv("../data/raw/criminalidad_anio_mes.csv", encoding="utf-8")
crim_comunas = pd.read_csv("../data/raw/criminalidad_comunas_anio.csv", encoding="utf-8")
homicidios_bm = pd.read_csv("../data/raw/homicidios_banco_mundial.csv", encoding="utf-8", skiprows=4)

print("Datasets cargados correctamente")

Datasets cargados correctamente


In [2]:
print("crim_mes:", crim_mes.shape)
print("crim_comunas:", crim_comunas.shape)
print("homicidios_bm:", homicidios_bm.shape)

crim_mes: (1586, 3)
crim_comunas: (2569, 4)
homicidios_bm: (266, 71)


**Verificación de la carga:** los tres datasets se cargaron sin errores. El número de filas coincide con las líneas de cada archivo (descontando el encabezado y, en el caso del Banco Mundial, las 4 líneas de metadatos omitidas con `skiprows=4`). El dataset del Banco Mundial presenta 71 columnas: 4 descriptivas, 66 de años (1960–2025) y una columna adicional generada por una coma sobrante al final de cada línea.

## Momento 3. Inspección de la estructura de los datasets

### Dataset 1: criminalidad por año y mes (`crim_mes`)

In [3]:
crim_mes.head()

,Fecha_hecho,Conducta,Cantidad_casos
0,2003-01,Extorsión,23
1,2003-02,Extorsión,15
2,2003-03,Extorsión,9
3,2003-04,Extorsión,11
4,2003-05,Extorsión,9


In [4]:
crim_mes.tail()

,Fecha_hecho,Conducta,Cantidad_casos
1581,2022-05,Hurto de semoviente,1
1582,2022-07,Hurto de semoviente,1
1583,2022-10,Hurto de semoviente,1
1584,2023-04,Hurto de semoviente,1
1585,2023-07,Hurto de semoviente,1


In [5]:
crim_mes.shape

(1586, 3)

In [6]:
crim_mes.columns

Index(['Fecha_hecho', 'Conducta', 'Cantidad_casos'], dtype='str')

In [7]:
crim_mes.dtypes

Fecha_hecho         str
Conducta            str
Cantidad_casos    int64
dtype: object

In [8]:
crim_mes.info()


<class 'pandas.DataFrame'>
RangeIndex: 1586 entries, 0 to 1585
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Fecha_hecho     1586 non-null   str  
 1   Conducta        1586 non-null   str  
 2   Cantidad_casos  1586 non-null   int64
dtypes: int64(1), str(2)
memory usage: 37.3 KB


In [9]:
crim_mes.describe()

,Cantidad_casos
count,1586.000000
mean,328.787516
std,660.281616
min,1.000000
25%,44.250000
50%,124.000000
75%,341.000000
max,5069.000000


In [10]:
crim_mes.describe(include='object')

C:\Users\panes\AppData\Local\Temp\ipykernel_32048\1502936883.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  crim_mes.describe(include='object')


,Fecha_hecho,Conducta
count,1586,1586
unique,251,7
top,2003-01,Homicidio
freq,7,251


In [11]:
print("Primera fecha:", crim_mes["Fecha_hecho"].min())
print("Última fecha:", crim_mes["Fecha_hecho"].max())

Primera fecha: 2003-01
Última fecha: 2023-11


### Dataset 2: criminalidad en comunas por año (`crim_comunas`)

In [12]:
crim_comunas.head()

,Fecha_hecho,Conducta,Codigo_comuna,Cantidad_casos
0,2003,Extorsión,3,1
1,2003,Extorsión,SIN DATO,143
2,2004,Extorsión,10,2
3,2004,Extorsión,13,1
4,2004,Extorsión,16,2


In [13]:
crim_comunas.tail()

,Fecha_hecho,Conducta,Codigo_comuna,Cantidad_casos
2564,2022,Hurto de semoviente,50,1
2565,2022,Hurto de semoviente,60,1
2566,2022,Hurto de semoviente,80,2
2567,2023,Hurto de semoviente,60,1
2568,2023,Hurto de semoviente,7,1


In [14]:
crim_comunas.shape

(2569, 4)

In [15]:
crim_comunas.columns

Index(['Fecha_hecho', 'Conducta', 'Codigo_comuna', 'Cantidad_casos'],
      dtype='str')

In [16]:
crim_comunas.dtypes

Fecha_hecho       int64
Conducta            str
Codigo_comuna       str
Cantidad_casos    int64
dtype: object

In [17]:
crim_comunas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2569 entries, 0 to 2568
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Fecha_hecho     2569 non-null   int64
 1   Conducta        2569 non-null   str  
 2   Codigo_comuna   2569 non-null   str  
 3   Cantidad_casos  2569 non-null   int64
dtypes: int64(2), str(2)
memory usage: 80.4 KB


In [18]:
crim_comunas.describe()

,Fecha_hecho,Cantidad_casos
count,2569.000000,2569.000000
mean,2013.340599,202.570261
std,6.003211,699.555102
min,2003.000000,1.000000
25%,2008.000000,14.000000
50%,2013.000000,47.000000
75%,2019.000000,153.000000
max,2023.000000,14195.000000


In [19]:
crim_comunas.describe(include='object')

C:\Users\panes\AppData\Local\Temp\ipykernel_32048\244699976.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  crim_comunas.describe(include='object')


,Conducta,Codigo_comuna
count,2569,2569
unique,7,43
top,Hurto a persona,13
freq,440,108


In [20]:
print("Primer año:", crim_comunas["Fecha_hecho"].min())
print("Último año:", crim_comunas["Fecha_hecho"].max())

Primer año: 2003
Último año: 2023


### Dataset 3: homicidios intencionales, Banco Mundial (`homicidios_bm`)


In [21]:
homicidios_bm.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,AFE,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,13.100000,12.500000,12.300000,11.700000,12.500000,12.7,12.4,NaN,NaN,NaN
2,Afganistán,AFG,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,6.792021,6.733248,7.163967,6.578109,4.032458,NaN,NaN,NaN,NaN,NaN
3,NaN,AFW,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,15.700000,15.000000,14.400000,13.700000,13.100000,12.6,11.9,NaN,NaN,NaN
4,Angola,AGO,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
homicidios_bm.tail()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
261,Kosovo,XKX,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,2.714749,2.489052,1.367489,2.112575,1.894977,NaN,NaN,NaN,NaN,NaN
262,"Yemen, Rep. del",YEM,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,Sudáfrica,ZAF,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,35.284016,35.865763,35.787476,32.977567,40.942983,43.720255,NaN,NaN,NaN,NaN
264,Zambia,ZMB,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
265,Zimbabwe,ZWE,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4.882120,5.173080,5.023544,6.216288,6.758331,NaN,NaN,NaN,NaN


In [23]:
homicidios_bm.shape

(266, 71)

In [24]:
homicidios_bm.columns

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024', '2025', 'Unnamed: 70'],
      dtype='str')

In [25]:
homicidios_bm.dtypes

Country Name          str
Country Code          str
Indicator Name        str
Indicator Code        str
1960              float64
                   ...   
2022              float64
2023              float64
2024              float64
2025              float64
Unnamed: 70       float64
Length: 71, dtype: object

In [26]:
homicidios_bm.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    264 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            0 non-null      float64
 5   1961            0 non-null      float64
 6   1962            0 non-null      float64
 7   1963            0 non-null      float64
 8   1964            0 non-null      float64
 9   1965            0 non-null      float64
 10  1966            0 non-null      float64
 11  1967            0 non-null      float64
 12  1968            0 non-null      float64
 13  1969            0 non-null      float64
 14  1970            0 non-null      float64
 15  1971            0 non-null      float64
 16  1972            0 non-null      float64
 17  1973            0 non-null      float64
 18  1

In [28]:
homicidios_bm.describe()

,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,149.000000,147.000000,151.000000,147.000000,134.000000,123.000000,101.000000,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.916301,7.418934,6.965127,6.907980,7.727898,8.299010,8.161892,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.872480,10.390563,9.811669,9.350369,10.176634,12.312002,12.374531,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.196286,0.159734,0.211657,0.177931,0.066588,0.069145,0.069095,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.215804,1.136858,0.969190,1.005808,1.015425,0.980352,0.876191,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.058363,2.900000,2.900000,2.963227,3.061466,2.742223,2.627800,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.300000,8.700000,7.921524,8.664888,11.112802,10.735610,10.400000,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,63.764040,53.795877,47.462677,51.818725,51.943814,76.340040,64.160144,NaN,NaN,NaN


In [29]:
homicidios_bm.describe(include='object')

C:\Users\panes\AppData\Local\Temp\ipykernel_32048\2303979262.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  homicidios_bm.describe(include='object')


,Country Name,Country Code,Indicator Name,Indicator Code
count,264,266,266,266
unique,264,266,1,1
top,Aruba,ABW,Homicidios intencionales (por cada 100.000 hab...,VC.IHR.PSRC.P5
freq,1,1,266,266


### Tabla de diagnóstico estructural

| Dataset | Filas | Columnas | Tipos de datos | Observaciones |
|---|---|---|---|---|
| `crim_mes` | 1586 | 3 | `Fecha_hecho`: str · `Conducta`: str · `Cantidad_casos`: int64 | Sin valores nulos. 7 conductas y 251 meses seguidos, de 2003-01 a 2023-11: no hay datos de 2024 ni de 2025, y 2023 está incompleto. `Fecha_hecho` está almacenada como texto y no como fecha. `Cantidad_casos` va de 1 a 5069, sin valores negativos ni ceros: los meses sin casos no tienen fila (faltan 171 de las 1757 combinaciones posibles de mes y conducta). Alta dispersión (media 328,8; desviación 660,3) porque mezcla conductas con magnitudes muy distintas. |
| `crim_comunas` | 2569 | 4 | `Fecha_hecho`: int64 · `Conducta`: str · `Codigo_comuna`: str · `Cantidad_casos`: int64 | Sin valores nulos según `info()`, pero `Codigo_comuna` contiene el texto `SIN DATO` en 76 registros, que es un faltante no detectado por `info()`. Cubre de 2003 a 2023. `Fecha_hecho` contiene solo el año y es entera, a diferencia del dataset 1, donde es texto con año y mes. `Codigo_comuna` es texto por la presencia de `SIN DATO` y tiene 43 valores únicos porque los mismos 21 códigos de zona (16 comunas y 5 corregimientos) aparecen escritos de dos formas, como entero (`1`) y como decimal (`1.0`), además de `SIN DATO`. El formato decimal aparece en todos los años, de 2003 a 2023, y corresponde a la conducta ____. `Cantidad_casos` va de 1 a 14.195. Contiene las mismas 7 conductas del dataset 1. |
| `homicidios_bm` | 266 | 71 | 4 columnas str · 67 columnas float64 | Formato ancho: una columna por año, de 1960 a 2025. Requiere `skiprows=4` para omitir los metadatos iniciales. 33 columnas completamente vacías: 1960–1989, 2024, 2025 y `Unnamed: 70`, generada por una coma sobrante. La cobertura disminuye en los años recientes (147 países en 2020 y 101 en 2023). Mezcla países y regiones; 2 filas sin `Country Name` (`AFE` y `AFW`, agregados regionales de África). `Indicator Name` e `Indicator Code` son constantes. Mide una tasa por 100.000 habitantes a nivel nacional, no cantidad de casos en Medellín. Para Colombia hay datos de 2020 a 2023 (entre 22,6 y 25,6 homicidios por cada 100.000 habitantes) y faltan 2024 y 2025. |

In [30]:
crim_comunas["Codigo_comuna"].value_counts()

Codigo_comuna
13          108
7           108
60          107
80          107
16          106
5           105
10          104
8           104
11          104
14          104
4           103
12          103
6           103
15          102
9           102
1           101
2            99
3            98
90           85
SIN DATO     76
70           74
50           38
1.0          21
2.0          21
3.0          21
4.0          21
5.0          21
6.0          21
7.0          21
8.0          21
9.0          21
10.0         21
11.0         21
12.0         21
13.0         21
14.0         21
15.0         21
16.0         21
60.0         21
70.0         21
80.0         21
90.0         19
50.0         10
Name: count, dtype: int64

In [31]:
crim_comunas.loc[crim_comunas["Codigo_comuna"].str.endswith(".0"), "Fecha_hecho"].unique()

array([2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
       2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023])

In [32]:
homicidios_bm.loc[homicidios_bm["Country Code"] == "COL", ["Country Name", "2020", "2021", "2022", "2023", "2024", "2025"]]

,Country Name,2020,2021,2022,2023,2024,2025
45,Colombia,22.642703,25.580127,24.948808,24.913442,NaN,NaN


In [33]:
homicidios_bm.loc[homicidios_bm["Country Name"].isna(), ["Country Name", "Country Code"]]

,Country Name,Country Code
1,NaN,AFE
3,NaN,AFW


In [34]:
crim_comunas.loc[crim_comunas["Codigo_comuna"].str.endswith(".0"), "Conducta"].unique()

<StringArray>
['Homicidio']
Length: 1, dtype: str